# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant dataset organizes data into record sets. We'll inspect available record sets, and for each one, display their fields and columns using their `@id` attributes.

In [ ]:
# List all record sets and their fields with @id references
record_set_ids = []
for record_set in metadata.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  name: {record_set.name}")
    print(f"  description: {getattr(record_set, 'description', '')}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.id} (name={field.name}, type={field.data_type})")
        # If the field has columns, show their @id
        columns = getattr(field, 'columns', None)
        if columns:
            print("      Columns:")
            for col in columns:
                print(f"        * Column @id: {col.id} (name={col.name})")
    print()
    record_set_ids.append(record_set.id)

if not record_set_ids:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we will load the first available record set from those found above.

In [ ]:
# Extract data from each record set
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet {record_set_id} with columns:")
            print(df.columns.tolist())
            display(df.head())
        else:
            print(f"No records found for RecordSet: {record_set_id}")
    # Select the first loaded DataFrame for further analyses
    if dataframes:
        selected_record_set_id = list(dataframes.keys())[0]
    else:
        selected_record_set_id = None
else:
    selected_record_set_id = None

# Print selected record set
print(f"Selected record set for further analysis: {selected_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Choose a numeric field from the selected record set (for example, a molecular weight or abundance field).
- Filter rows based on a threshold.
- Normalize numeric values.
- Optionally, group by a categorical field.

In [ ]:
import numpy as np

df = dataframes[selected_record_set_id] if selected_record_set_id else pd.DataFrame()

if not df.empty:
    # Try to pick a likely numeric field for demonstration
    sample_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'fi' and not df[col].isnull().all()]
    if not sample_numeric_fields:
        # If no float/integer found, try to convert any numeric column
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        sample_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'fi']

    if sample_numeric_fields:
        numeric_field = sample_numeric_fields[0]
        print(f"Using numeric field '{numeric_field}' for analysis.")
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 0
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Optionally group by a categorical field
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            group_field = None
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in this record set. EDA steps skipped.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the numeric field analyzed above, and, if grouping was possible, show a barplot of means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        group_means = group_means[:20]  # Show top 20 groups for readability
        sns.barplot(y=group_means.index, x=group_means.values)
        plt.title(f"Mean {numeric_field} by {group_field} (top 20)")
        plt.xlabel(numeric_field)
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the Croissant dataset and listed all available record sets and fields using their `@id`s.
- We extracted tabular data from the first record set and performed simple data exploration, including filtering, normalization, and optional grouping.
- Visualizations showed distributional properties and possible groupwise patterns.

Continue to explore other record sets or fields using their `@id` attributes to deepen your understanding of this rich mass spectrometry dataset.